# Images, metadata and segmentation

> **Notebook role:** production-style image intake, dataset examination and segmentation sequence.

## 1. Examine the dataset before loading pixels

Start from paths and acquisition metadata. Confirm file type, dimensions,
channel order and timepoint order before choosing an image reader.

In [ ]:
from pathlib import Path
import pandas as pd

from microscopy_workflows import describe_paths

image_paths = sorted(Path("data/images").glob("*"))
pd.DataFrame(describe_paths(image_paths))

## 2. Inspect shape and intensity range

The array shape determines whether the segmentation runs on a 2D image, a stack
of independent planes or a connected 3D volume. Robust percentiles reveal
whether enhancement is useful.

In [ ]:
from tifffile import imread
from nuclear_imaging_core.measurements import describe_image

if not image_paths:
    raise FileNotFoundError("Place microscopy images in data/images before running this notebook")
image = imread(image_paths[0])
describe_image(image)

## 3. Segment with explicit pixel thresholds

The same configuration object selects StarDist for fluorescence images or a
multi-level threshold workflow for 3D intensity volumes.

In [ ]:
from nuclear_imaging_core.segmentation import SegmentationConfig, segment_frame
from microscopy_workflows import load_config

config = SegmentationConfig(**load_config("segmentation_2d.json"))
labels, foreground = segment_frame(image, config)
{"objects": int(labels.max()), "foreground_pixels": int(foreground.sum())}

## 4. Save the stage output

Persist the source image, label image and foreground mask together. The next
notebook consumes this named artifact, avoiding hidden notebook state.

In [ ]:
import numpy as np

output_path = Path("outputs/segmentation.npz")
output_path.parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(output_path, image=image, labels=labels, foreground=foreground)
output_path

## Representative results

Inspect source pixels, intermediate masks and final object labels together.
The relevant checks differ with the acquisition and segmentation target.

### Brightfield spheroids: connected objects and expanded labels

![Brightfield spheroid segmentation progression](../assets/results/segmentation-progression.png)

### RGB transmitted-light spheroids: colour-derived signal and seeds

![RGB spheroid segmentation stages](../assets/results/rgb-spheroid-segmentation.png)

### Confocal z-stack projection: nuclear labels and watershed separation

![Fluorescence nuclear segmentation stages](../assets/results/fluorescence-nucleus-segmentation.png)